# BabelBetes Data-Output Validation

Checks the *standardized output* (CGM, basal, bolus, carbs) — not the extraction code — for whether the harmonized data is physiologically plausible. This is the automated, run-once-over-the-output part of the paper's V&V section.

**The idea:**   
 1. load the store
 2. compute **wide** per-patient metric tables (each data type separately, correlations handled later)
 3. day-level metrics are aggreagted to the patient
 4. Check patient aggreagtes against declarative `BOUNDS` lists that define thresholds for raising flags.
 5. Bounds also defines if raised flags are **hard** (a defect → fix, or approve with a documented reason) or **soft** (unusual but possible → the challenge set, investigated case by case). 


## 1. Load Data

Start with ReplaceBG and Loop

In [1]:
import os
import numpy as np
import pandas as pd
from functools import reduce
import babelbetes.data_store as ds
from matplotlib import pyplot as plt
import seaborn as sns

# Resolve the store whether the notebook runs from the repo root or from notebooks/

BASE_PATH = os.path.join(os.getcwd(),'../data/out')

# --- Bounds / limits (single source of truth for this notebook) ---
CGM_MIN, CGM_MAX = 40, 400          # CF-2 hard physiologic bounds (mg/dL)
COV_THRESHOLD = 0.70                # PLACEHOLDER soft bound: flag patients below this coverage


In [2]:
STUDIES = ['Loop', 'ReplaceBG']
store = ds.load(BASE_PATH, studies=STUDIES)

results = []
for data_type,df in store.items():
    results.append((data_type,df.groupby('study_name').size()))
results = [df.reset_index().assign(data_type=data_type) for (data_type, df) in results]
pd.concat(results).sort_values(['study_name','data_type'])

,study_name,0,data_type
0,Loop,919,age
0,Loop,48080493,basal
0,Loop,2422370,bolus
0,Loop,1467850,carbs
0,Loop,89203827,cgm
1,ReplaceBG,196,age
1,ReplaceBG,316138,basal
1,ReplaceBG,291662,bolus
1,ReplaceBG,150060,carbs
1,ReplaceBG,12034638,cgm


## Bounds → flags → hard/soft sets

Metric tables are wide: one row per patient, one column per metric. 

To find the patients that need attention, we check each metric column against a short list of rules `BOUNDS`. Every line in the BOUNDS defines the metrics, flagging rule, a description of the rule and if the flag is hard or soft.

In [21]:
def evaluate(metrics, bounds, keys=['study_name', 'patient_id']):
    """Wide metric table + bounds spec -> the one long flags table.

    Grain-agnostic (pass different `keys` for the study grain) and table-agnostic:
    bounds whose metric column is absent from `metrics` are skipped, so a single
    BOUNDS list can cover every metric table.
    """
    cols = keys + ['metric', 'value', 'bound', 'rule', 'disposition']
    hits = []
    for col, bound, flag_if, rule, disposition in bounds:
        if col not in metrics.columns:
            continue
        mask = flag_if(metrics[col]) & metrics[col].notna()   # notna: never flag a missing metric
        h = metrics.loc[mask, keys].copy()
        h['metric'] = col
        h['value'] = metrics.loc[mask, col].to_numpy()
        h['bound'] = bound
        h['rule'] = rule
        h['disposition'] = disposition
        hits.append(h)
    return (pd.concat(hits, ignore_index=True) if hits else pd.DataFrame(columns=cols))[cols]


# EXAMPLE
# --- tiny example: 2 studies × 3 patients, 2 metrics (no real data needed) ---
example = pd.DataFrame({
    'study_name': ['StudyA', 'StudyA', 'StudyA', 'StudyB', 'StudyB', 'StudyB'],
    'patient_id': ['a', 'b', 'c', 'd', 'e', 'f'],
    'coverage':   [0.99, 0.55, 0.98, 0.97, 0.60, 0.99],   # fraction of days with data
    'value':      [42.0,  8.0, 250.0, 30.0, 90.0,  0.5],  # mean daily insulin (U)
})
display(example)

EXAMPLE_BOUNDS = [
    # column,     H/S, flag-if predicate,             rule text,               disposition
    ('coverage', 'S', lambda s: s < 0.6,             'coverage < 75%',       'investigate'),
    ('value', 'H',    lambda s: ~s.between(1, 150),   'TDD outside [1, 150]', 'fix/approve'),
]

# evaluate() checks every column against its rule and stacks the hits into one long table.
print('flagged patient rows:')
results = evaluate(example, EXAMPLE_BOUNDS)
display(results)

# per study summary
display(results.groupby(['study_name', 'bound', 'rule']).size().rename('cases').reset_index())


,study_name,patient_id,coverage,value
0,StudyA,a,0.99,42.0
1,StudyA,b,0.55,8.0
2,StudyA,c,0.98,250.0
3,StudyB,d,0.97,30.0
4,StudyB,e,0.60,90.0
5,StudyB,f,0.99,0.5


flagged patient rows:


,study_name,patient_id,metric,value,bound,rule,disposition
0,StudyA,b,coverage,0.55,S,coverage < 75%,investigate
1,StudyA,c,value,250.00,H,"TDD outside [1, 150]",fix/approve
2,StudyB,f,value,0.50,H,"TDD outside [1, 150]",fix/approve


,study_name,bound,rule,cases
0,StudyA,H,"TDD outside [1, 150]",1
1,StudyA,S,coverage < 75%,1
2,StudyB,H,"TDD outside [1, 150]",1


## Metric Calculations

### CGM

In [22]:
def num_out_of_range(ds_cgm: pd.Series, low=40, high=400):
    return (~ds_cgm.between(low, high, inclusive='both')).sum()

def daily_sample_avg(ds: pd.Series):
    return ds.groupby(ds.dt.date).count().mean()

def uptime(ds_datetime: pd.Series, interval='15min'):
    #fraction of day covered with cgm data 
    bins_per_day = pd.Timedelta('1D') / pd.Timedelta(interval)  # e.g. 48 for 30min
    floored = ds_datetime.dt.floor(interval)
    date = ds_datetime.dt.date
    frac = floored.groupby(date, sort=False).nunique() / bins_per_day
    return frac

def day_coverage(ds_datetime: pd.Series):
    #fraction of days covered between first and last data point
    num_days_range =  (ds_datetime.max().normalize() - ds_datetime.min().normalize()).days + 1
    frac_present = ds_datetime.dt.normalize().nunique() / num_days_range
    return frac_present

#calcualte
cgm = store['cgm'].copy()
cgm_presence_patiently = cgm.groupby(['study_name','patient_id']).agg(
    day_coverage  = ('datetime', day_coverage),                         #check for <75%
    out_of_range  = ('cgm',num_out_of_range),                           #hard check >0
    CV_glucose    = ('cgm', lambda ds: ds.std()/ds.mean()),             #check for <10%, > 50%
    uptime        = ('datetime',lambda ds_dt: uptime(ds_dt).mean()),    #check for < 80%
    ).reset_index()
display(cgm_presence_patiently.head())

CGM_BOUNDS = [
    ('out_of_range', 'H', lambda s: s > 0,                  'out of range values > 0', 'fix/approve'),
    ('day_coverage', 'S', lambda s: s < 0.75,               'day coverage < 75%',      'investigate'),
    ('uptime',       'S', lambda s: s < 0.80,               'cgm uptime < 80%',        'investigate'),
    ('CV_glucose',   'S', lambda s: ~s.between(0.10, 0.50), 'CV outside [0.10, 0.50]', 'investigate'),
]

cgm_flags = evaluate(cgm_presence_patiently, CGM_BOUNDS)
display(cgm_flags.head())
display(cgm_flags.groupby(['bound', 'rule']).size().rename('cases').reset_index())

# #draw
# plt.figure(figsize=(8,3))
# sns.ecdfplot(cgm_presence_patiently,x='CV_glucose',hue='study_name')

# plt.figure(figsize=(8,3))
# sns.ecdfplot(cgm_presence_patiently,x='uptime',hue='study_name')

# plt.figure(figsize=(8,3))
# sns.ecdfplot(cgm_presence_patiently,x='day_coverage',hue='study_name')


,study_name,patient_id,day_coverage,out_of_range,CV_glucose,uptime
0,Loop,10,0.941624,0,0.352802,0.966813
1,Loop,100,0.986486,0,0.448961,0.958305
2,Loop,1000,0.997625,0,0.441270,0.958929
3,Loop,1001,0.957265,0,0.424856,0.971633
4,Loop,1002,1.000000,0,0.330089,0.982715


,study_name,patient_id,metric,value,bound,rule,disposition
0,Loop,1172,day_coverage,0.745161,S,day coverage < 75%,investigate
1,Loop,1204,day_coverage,0.666667,S,day coverage < 75%,investigate
2,Loop,1205,day_coverage,0.726619,S,day coverage < 75%,investigate
3,Loop,151,day_coverage,0.674487,S,day coverage < 75%,investigate
4,Loop,274,day_coverage,0.289855,S,day coverage < 75%,investigate


,bound,rule,cases
0,S,"CV outside [0.10, 0.50]",3
1,S,cgm uptime < 80%,21
2,S,day coverage < 75%,23


## TDD

In [ ]:
from babelbetes.tdd import calculate_daily_basal_dose, calculate_daily_bolus_dose

def build_daily_tdd(store, keys=['study_name', 'patient_id']):
    """Daily TDD per (study, patient, date) — the intermediate we roll up to the patient.

    Reuses tdd.py's daily helpers, grouped by study+patient (NOT patient alone, as
    calculate_tdd does) so identical patient_ids across studies never collide.
    """
    basal = store['basal'].groupby(keys, observed=True).apply(calculate_daily_basal_dose, include_groups=False)
    bolus = store['bolus'].groupby(keys, observed=True).apply(calculate_daily_bolus_dose, include_groups=False)
    tdd = basal.merge(bolus, how='outer', on=keys + ['date']).reset_index()
    tdd['bolus'] = tdd['bolus'].fillna(0)                       # no bolus that day = 0 U
    tdd['tdd'] = tdd[['basal', 'bolus']].sum(axis=1, min_count=1)
    return tdd

tdd_daily = build_daily_tdd(store)
tdd_daily['tdd_ratio'] = tdd_daily['basal']/tdd_daily['bolus']

# roll the daily TDD up to the PATIENT grain: count anomalous days + basal/bolus split
tdd_patient = tdd_daily.groupby(['study_name', 'patient_id']).agg(
    frac_tdd_eq0        = ('tdd',   lambda s: (s == 0).mean()),
    frac_tdd_bolus_eq0  = ('bolus', lambda s: (s == 0).mean()),
    frac_tdd_basal_eq0  = ('basal', lambda s: (s == 0).mean()),                               
    frac_tdd_lt1        = ('tdd',   lambda s: s.between(0, 1, inclusive='neither').mean()),   
    frac_tdd_gt100      = ('tdd',   lambda s: (s > 200).mean()),                              
    tdd_ratio           = ('tdd_ratio',   'median')
).reset_index()

# TDD bounds — patient grain. Count thresholds are tunable (start at "any occurrence").
TDD_BOUNDS = [
    ('frac_tdd_eq0',   'H', lambda s: s > 0.1,               'Zero TDD > 10%',            'fix/approve'),
    ('frac_tdd_bolus_eq0',   'H', lambda s: s > 0.1,         'Zero Bolus Days >10%',          'fix/approve'),
    ('frac_tdd_basal_eq0',   'H', lambda s: s > 0.1,         'Zero Bolus Days >10%',          'fix/approve'),
    ('days_tdd_lt1',   'S', lambda s: s > 0,                 'has day(s) 0<TDD<1',            'investigate'),
    ('days_tdd_gt100', 'S', lambda s: s > 0,                 'has day(s) TDD>100',            'investigate'),
    ('tdd_ratio',      'S', lambda s: ~s.between(0.2, 0.8),  'basal/bolus outside [0.2,0.8]', 'investigate'),
]

tdd_flags = evaluate(tdd_patient, TDD_BOUNDS)   # same evaluate(), default (patient) keys

In [24]:
display(tdd_flags.sample(5))
display(tdd_flags.groupby(['study_name','bound', 'rule']).size().rename('cases').reset_index())

,study_name,patient_id,metric,value,bound,rule,disposition
21,Loop,1089,frac_tdd_eq0,0.136646,H,Zero TDD > 10%,fix/approve
120,Loop,435,frac_tdd_eq0,0.282297,H,Zero TDD > 10%,fix/approve
629,Loop,121,tdd_ratio,2.147698,S,"basal/bolus outside [0.2,0.8]",investigate
540,Loop,1094,tdd_ratio,2.173119,S,"basal/bolus outside [0.2,0.8]",investigate
1003,Loop,669,tdd_ratio,1.504484,S,"basal/bolus outside [0.2,0.8]",investigate


,study_name,bound,rule,cases
0,Loop,H,Zero Bolus Days >10%,249
1,Loop,H,Zero TDD > 10%,202
2,Loop,S,"basal/bolus outside [0.2,0.8]",763
3,ReplaceBG,H,Zero Bolus Days >10%,14
4,ReplaceBG,H,Zero TDD > 10%,4
5,ReplaceBG,S,"basal/bolus outside [0.2,0.8]",146
